**Grupo 4** | AiDAPT03  
Projeto de grupo desenvolvido no âmbito do curso AiDAPT da Cegid Academy

# Projeto: Análise de RH, Predição de Mudança de Emprego
## Fase 3: Data Preparation (Preparação dos Dados)

**Metodologia:** CRISP-DM | **Fase:** 3 de 5  
**Objetivo:** Limpar os dados, tratar os valores em falta, converter as variáveis para formatos adequados ao modelo, e dividir em conjuntos de treino e teste.

---
## 1. Importações e Carregamento

In [2]:
""" Importamos as bibliotecas necessárias para toda a fase de preparação.
O pandas trata da manipulação tabular, o matplotlib e o seaborn permitem
visualizar o efeito das transformações, e o io permite construir um objecto
de ficheiro em memória a partir de texto, necessário para a função de
carregamento. O sklearn fornece as ferramentas de divisão treino/teste e
encoding, e o imblearn o SMOTE para balanceamento de classes."""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import io
import warnings
import os
from sklearn.model_selection import train_test_split                 # divide os dados em treino e teste
from sklearn.preprocessing import LabelEncoder                       # converte categorias em inteiros
from imblearn.over_sampling import SMOTE                             # balanceamento sintético da classe minoritária

pd.set_option('display.max_rows', None)     # remove o limite de linhas no output do pandas
pd.set_option('display.max_columns', None)  # remove o limite de colunas no output do pandas
pd.set_option('display.width', None)        # remove o limite de largura do output

warnings.filterwarnings('ignore')                                    # suprime avisos não críticos durante a execução
sns.set_theme(style='whitegrid', font_scale=1.1)                     # tema visual consistente em todos os gráficos

SEED = 42                                                            # semente aleatória para reprodutibilidade

# ATENÇÃO: actualizar este caminho se a pasta for movida ou o projeto aberto noutra máquina
BASE_DIR      = r"C:\Users\Utilizador\Desktop\CEGID\PYTHON\Projeto - Grupo 4\Projeto Final - SMOTE e GridSearchCV"
INPUT_DIR     = os.path.join(BASE_DIR, 'Data', 'input')              # pasta onde está o ficheiro CSV original
OUTPUT_DIR    = os.path.join(BASE_DIR, 'Data', 'output')             # pasta onde são guardados os ficheiros gerados
ANALYTICS_DIR = os.path.join(BASE_DIR, 'Data', 'analytics')          # pasta para ficheiros de análise intermédia

for d in [INPUT_DIR, OUTPUT_DIR, ANALYTICS_DIR]:                     # percorre as três pastas
    os.makedirs(d, exist_ok=True)                                    # cria a pasta se não existir; se já existir não dá erro

print('✅ Ambiente configurado!')

✅ Ambiente configurado!


In [5]:
# Construir o caminho completo para o ficheiro CSV usando os.path.join
# os.path.join garante que os separadores de pasta são correctos em qualquer sistema operativo
caminho = os.path.join(INPUT_DIR, 'Dados_RH_Mudanca_Emprego(in).csv')
def carregar_dados(caminho):
    """Carrega o CSV corrigindo linhas com aspas duplas externas.
    O ficheiro tem um problema específico: linhas cujo campo 'area_formacao'
    contém vírgulas ficam encapsuladas em aspas duplas externas, o que confunde
    o parser padrão do pandas. Esta função lê o ficheiro linha a linha,
    corrige esse problema e devolve um DataFrame limpo.
    """
    with open(caminho, 'rb') as f:                                           # abre em modo binário para controlar a codificação manualmente
        linhas = f.readlines()                                               # lê todas as linhas para uma lista

    linhas_fixas = []                                                        # lista onde ficam as linhas já corrigidas
    for linha in linhas:                                                     # percorre cada linha uma a uma
        txt = linha.decode('latin-1').rstrip('\r\n')                         # converte bytes em texto e remove o fim de linha
        if txt.startswith('"') and txt.endswith('"'):                        # detecta linhas encapsuladas em aspas externas
            txt = txt[1:-1].replace('""', '"')                               # remove as aspas externas e normaliza as internas
        linhas_fixas.append(txt)                                             # guarda a linha corrigida na lista

    return pd.read_csv(io.StringIO('\n'.join(linhas_fixas)))                 # reconstrói o texto corrigido e passa ao pandas como ficheiro virtual


# Carregar os dados com a função corrigida
# df_raw guarda o estado original intocado — útil para repor se uma transformação correr mal
# df é a cópia de trabalho onde todas as alterações são feitas
df_raw = carregar_dados(caminho)                                             # carrega o CSV já corrigido
df = df_raw.copy()                                                           # cria a cópia de trabalho

print(f'📊 Dataset carregado: {df.shape[0]:,} linhas x {df.shape[1]} colunas')

display(df.head())    

📊 Dataset carregado: 19,158 linhas x 14 colunas


,id_candidato,cidade,indice_desenvolvimento_cidade,genero,experiencia_relevante,inscricao_universidade,nivel_educacao,area_formacao,anos_experiencia,dimensao_empresa,tipo_empresa,anos_desde_ultimo_emprego,horas_formacao,procura_mudanca_emprego
0,760695,Zona_049,0.910,NaN,Com experiência relevante,Não inscrito,Licenciatura,"Ciência, Tecnologia, Engenharia e Matemática",6 anos,500 a 999,Empresa privada,1 ano,21,Não
1,789493,Zona_006,0.920,Feminino,Com experiência relevante,Não inscrito,Mestrado,Humanidades,Mais de 20 anos,100 a 500,Startup financiada,2 anos,74,Não
2,211612,Zona_007,0.924,NaN,Com experiência relevante,Não inscrito,Licenciatura,"Ciência, Tecnologia, Engenharia e Matemática",9 anos,10 a 49,Empresa privada,1 ano,94,Não
3,521686,Zona_065,0.624,Masculino,Com experiência relevante,Não inscrito,Mestrado,"Ciência, Tecnologia, Engenharia e Matemática",15 anos,10000 ou mais,Empresa privada,1 ano,75,Não
4,146611,Zona_030,0.698,Masculino,Sem experiência relevante,Não inscrito,Mestrado,"Ciência, Tecnologia, Engenharia e Matemática",12 anos,500 a 999,ONG,1 ano,157,Não


---
## 2. Remover Duplicados e Colunas Desnecessárias

In [6]:
""" Removemos linhas duplicadas antes de qualquer outra transformação.
Duplicados inflam o peso de certos registos no treino e podem fazer
com que o mesmo registo apareça em treino e em teste em simultâneo,
o que falseia as métricas ao avaliar dados que o modelo já viu."""

# Remover duplicados
antes = len(df)                                                      # guarda o número de linhas antes da remoção
df = df.drop_duplicates()                                            # remove linhas completamente iguais
print(f'Duplicados removidos: {antes - len(df)}')                    # mostra quantas linhas foram removidas

"""Removemos 'id_candidato' porque é um identificador único sem qualquer
poder preditivo; incluí-lo levaria o modelo a aprender padrões espúrios.
Removemos 'cidade' porque tem centenas de categorias distintas e o
'indice_desenvolvimento_cidade' já representa essa dimensão de forma
numérica e contínua, com muito menos ruído para o modelo."""
df = df.drop(columns=['id_candidato', 'cidade'])                     # remove as duas colunas do DataFrame
print(f'Colunas após remoção: {df.columns.tolist()}')                # lista as colunas que ficaram

Duplicados removidos: 0
Colunas após remoção: ['indice_desenvolvimento_cidade', 'genero', 'experiencia_relevante', 'inscricao_universidade', 'nivel_educacao', 'area_formacao', 'anos_experiencia', 'dimensao_empresa', 'tipo_empresa', 'anos_desde_ultimo_emprego', 'horas_formacao', 'procura_mudanca_emprego']


### Análise

Não foram encontrados duplicados, o que confirma o que já tínhamos identificado
no notebook anterior. O dataset mantém os 19 158 registos originais.

Após a remoção de **id_candidato** (identificador sem valor preditivo) e
**cidade** (centenas de categorias substituídas pelo **indice_desenvolvimento_cidade**),
o dataset passa a ter 12 colunas de trabalho.

---
## 3. Tratar a Variável Alvo

In [7]:
"""Verificamos quantos registos não têm a variável alvo preenchida.
É necessário fazer esta verificação antes de criar a coluna binária
para não propagar NaN de forma silenciosa para a coluna 'target'."""

print(f'Registos sem variável alvo: {df["procura_mudanca_emprego"].isnull().sum()}')
# se o valor for 0, podemos avançar com segurança

""" Removemos os registos sem variável alvo porque não é possível treinar
um modelo supervisionado sem conhecer a resposta correcta de cada observação.
Imputar a variável alvo introduziria um viés directo nos resultados,
pois estaríamos a avaliar o modelo com respostas fabricadas."""

df = df[df['procura_mudanca_emprego'].notna()].copy()
# .notna() selecciona apenas as linhas onde a variável alvo está preenchida
# .copy() evita o aviso de "SettingWithCopyWarning" do pandas

print(f'Registos restantes: {len(df):,}')                           # confirma quantos registos ficaram

""" Convertemos a variável alvo para binário com 1 para "Sim" e 0 para "Não".
Esta representação é exigida pelos algoritmos do scikit-learn e simplifica
o cálculo de métricas como precisão, recall, F1 e AUC-ROC, que assumem
que a classe positiva tem valor 1."""

df['target'] = (df['procura_mudanca_emprego'] == 'Sim').astype(int)
# (df['procura_mudanca_emprego'] == 'Sim') devolve True/False por linha
# .astype(int) converte True para 1 e False para 0

df = df.drop(columns=['procura_mudanca_emprego'])                    # remove a coluna original em texto

print('\nDistribuição da variável alvo:')
print(df['target'].value_counts().rename({1: 'Sim (1)', 0: 'Não (0)'}))
# mostra quantos registos pertencem a cada classe após a conversão

Registos sem variável alvo: 0
Registos restantes: 19,158

Distribuição da variável alvo:
target
Não (0)    14381
Sim (1)     4777
Name: count, dtype: int64


### Análise

Não existem registos sem variável alvo, o que significa que todos os 19 158
candidatos têm resposta preenchida e nenhum registo foi removido.

A variável alvo foi convertida para binário: 14 381 registos com valor 0 (Não)
e 4 777 com valor 1 (Sim), confirmando o desequilíbrio de classes já identificado
no notebook anterior (75.1% vs 24.9%).

---
## 4. Tratar Valores em Falta

In [8]:
""" Verificamos o estado dos nulos após a remoção dos registos sem variável alvo.
Este passo serve de ponto de controlo antes de definir a estratégia de
imputação, confirmando exactamente quais as colunas que ainda precisam
de tratamento e quantos valores faltam em cada uma."""

print('Valores em falta antes do tratamento:')
print(df.isnull().sum()[df.isnull().sum() > 0].sort_values(ascending=False))
# .isnull().sum() conta os nulos por coluna
# [df.isnull().sum() > 0] filtra apenas as colunas com pelo menos um nulo
# .sort_values(ascending=False) ordena da coluna com mais nulos para a com menos

Valores em falta antes do tratamento:
tipo_empresa                 6140
dimensao_empresa             5938
genero                       4508
area_formacao                2813
nivel_educacao                460
anos_desde_ultimo_emprego     423
inscricao_universidade        386
anos_experiencia               65
dtype: int64


### Análise

Confirmamos 8 colunas com valores em falta, consistentes com o que identificámos
no notebook anterior. As mais críticas continuam a ser
<span style="color: red; font-weight: bold;">tipo_empresa</span> (6 140),
<span style="color: red; font-weight: bold;">dimensao_empresa</span> (5 938) e
<span style="color: red; font-weight: bold;">genero</span> (4 508). A estratégia
de tratamento para cada coluna é definida nas células seguintes.

In [9]:
"""Para as variáveis numéricas preenchemos com a mediana em vez da média.
A mediana é robusta a outliers: valores extremos em 'horas_formacao' ou
em 'indice_desenvolvimento_cidade' não distorcem o valor imputado,
ao contrário da média, que seria puxada pelos extremos observados."""

for col in ['indice_desenvolvimento_cidade', 'horas_formacao']:
    mediana = df[col].median()                                       # calcula a mediana da coluna
    n = df[col].isnull().sum()                                       # conta os nulos antes de preencher
    df[col] = df[col].fillna(mediana)                                # substitui os nulos pela mediana
    print(f'{col}: {n} nulos preenchidos com mediana ({mediana:.2f})')

""" Para as variáveis categóricas usamos 'Desconhecido' em vez de imputar
a categoria mais frequente. Esta abordagem preserva a informação de que
o dado estava em falta sem inventar um valor; permite ainda que o modelo
aprenda um padrão associado à ausência de resposta, que pode ser
informativo neste contexto de candidatos a emprego."""

cols_cat = ['genero', 'nivel_educacao', 'area_formacao',
            'tipo_empresa', 'dimensao_empresa',
            'inscricao_universidade', 'experiencia_relevante',
            'anos_experiencia', 'anos_desde_ultimo_emprego']
# lista de todas as variáveis categóricas que precisam de tratamento de nulos

for col in cols_cat:
    n = df[col].isnull().sum()                                       # conta os nulos antes de preencher
    df[col] = df[col].fillna('Desconhecido')                         # substitui os nulos por 'Desconhecido'
    if n > 0:                                                        # só imprime se havia nulos na coluna
        print(f'{col}: {n} nulos preenchidos com "Desconhecido"')

print(f'\nTotal de nulos após tratamento: {df.isnull().sum().sum()}')
# .sum().sum() soma todos os nulos de todas as colunas, deve ser 0 após o tratamento

indice_desenvolvimento_cidade: 0 nulos preenchidos com mediana (0.90)
horas_formacao: 0 nulos preenchidos com mediana (47.00)
genero: 4508 nulos preenchidos com "Desconhecido"
nivel_educacao: 460 nulos preenchidos com "Desconhecido"
area_formacao: 2813 nulos preenchidos com "Desconhecido"
tipo_empresa: 6140 nulos preenchidos com "Desconhecido"
dimensao_empresa: 5938 nulos preenchidos com "Desconhecido"
inscricao_universidade: 386 nulos preenchidos com "Desconhecido"
anos_experiencia: 65 nulos preenchidos com "Desconhecido"
anos_desde_ultimo_emprego: 423 nulos preenchidos com "Desconhecido"

Total de nulos após tratamento: 0


### Análise

Os valores em falta foram tratados com duas estratégias distintas.

Para as variáveis numéricas usámos a **mediana** porque é menos sensível
a valores extremos do que a média. Nenhuma das duas tinha nulos,
pelo que não foi necessária imputação: **indice_desenvolvimento_cidade**
(mediana 0.90) e **horas_formacao** (mediana 47.00) estavam já completas.

Para as variáveis categóricas usámos **"Desconhecido"** para não inventar
um valor, permitindo que o modelo aprenda um padrão associado à ausência
de resposta. As colunas tratadas foram
<span style="color: red; font-weight: bold;">tipo_empresa</span> (6 140),
<span style="color: red; font-weight: bold;">dimensao_empresa</span> (5 938),
<span style="color: red; font-weight: bold;">genero</span> (4 508),
<span style="color: red; font-weight: bold;">area_formacao</span> (2 813),
<span style="color: orange; font-weight: bold;">nivel_educacao</span> (460),
<span style="color: orange; font-weight: bold;">anos_desde_ultimo_emprego</span> (423),
<span style="color: orange; font-weight: bold;">inscricao_universidade</span> (386) e
**anos_experiencia** (65).

O total de nulos após o tratamento é 0, o que confirma que todas as colunas
estão completamente preenchidas e o dataset está pronto para as transformações
seguintes.

---
## 5. Engenharia de Variáveis (Feature Engineering)

A principal transformação criada foi a conversão da coluna **anos_experiencia**
de texto para número através da função **conv_exp()**. Esta variável estava
guardada como texto com intervalos como "6 anos" ou "Mais de 20 anos" e não
podia ser usada directamente nos modelos. A conversão para número ordinal
permite que os algoritmos interpretem correctamente a progressão da experiência
profissional.

Não foram criadas outras variáveis derivadas porque as features existentes
já cobrem as dimensões relevantes do problema: contexto geográfico, perfil
académico, perfil profissional e características da empresa.

In [10]:
""" A coluna 'anos_experiencia' contém intervalos textuais como "6 anos" ou
"Mais de 20 anos" e não pode ser usada directamente num modelo numérico.
A função abaixo extrai o inteiro do texto, trata os casos extremos
("<1" fica 0, "Mais de 20" fica 21) e devolve NaN para entradas que
não seguem o padrão esperado, incluindo a categoria "Desconhecido"."""

def conv_exp(v):
    if pd.isna(v):                                                   # se o valor for NaN, devolve NaN sem tentar converter
        return np.nan
    s = str(v).strip()                                               # converte para texto e remove espaços à volta
    if 'Mais de' in s or '>20' in s:                                 # "Mais de 20 anos" ou ">20" → valor máximo convencional
        return 21
    if '<' in s:                                                     # "Menos de 1 ano" ou "<1" → zero anos de experiência
        return 0
    if 'Desconhecido' in s:                                          # valor preenchido na fase de nulos → tratar como desconhecido
        return np.nan
    try:
        return int(s.split()[0])                                     # "6 anos" → divide por espaço → pega no primeiro elemento → converte para inteiro
    except:
        return np.nan                                                # se a conversão falhar por qualquer razão, devolve NaN

"""Aplicamos a conversão e preenchemos os NaN resultantes com a mediana
dos valores convertidos com sucesso. Usamos a mediana pela mesma razão
que nas outras variáveis numéricas: é menos influenciada pelos extremos
e mantém a coerência da estratégia de imputação em todo o dataset."""

df['anos_exp_num'] = df['anos_experiencia'].apply(conv_exp)          # aplica a função a cada linha da coluna
df['anos_exp_num'] = df['anos_exp_num'].fillna(df['anos_exp_num'].median())
# preenche os NaN resultantes da conversão com a mediana dos valores convertidos

print('Primeiros valores convertidos:')
print(df[['anos_experiencia', 'anos_exp_num']].drop_duplicates().head(10).to_string())
# drop_duplicates() evita mostrar linhas repetidas
# .to_string() garante que o output não é truncado

Primeiros valores convertidos:
   anos_experiencia  anos_exp_num
0            6 anos           6.0
1   Mais de 20 anos          21.0
2            9 anos           9.0
3           15 anos          15.0
4           12 anos          12.0
5           13 anos          13.0
12            1 ano           1.0
13           2 anos           2.0
15          10 anos          10.0
16           7 anos           7.0


### Análise

A conversão foi bem sucedida. Os valores textuais foram correctamente
convertidos para números inteiros: intervalos como "6 anos" ficaram com
o valor 6, "Mais de 20 anos" ficou com 21 e "1 ano" ficou com 1.

Os registos com valor "Desconhecido" em **anos_experiencia** devolveram NaN
após a conversão e foram preenchidos com a mediana dos valores convertidos
com sucesso. A coluna **anos_exp_num** está pronta para ser usada no modelo.

---
## 6. Redução de Dimensionalidade

Foram removidas duas colunas na secção 2 deste notebook:

- **id_candidato** é um identificador único sem qualquer poder preditivo.
  Incluí-lo levaria o modelo a aprender padrões espúrios (falsos/artificiais).
- **cidade** tem 123 categorias distintas, demasiadas para ser usada
  directamente num modelo. O **indice_desenvolvimento_cidade** já representa
  essa dimensão de forma numérica e contínua, com muito menos ruído.

Não foi aplicado PCA porque o número de features após encoding (11 para
Label Encoding, 41 para One-Hot Encoding) é gerível e a interpretabilidade
das features originais é importante para o contexto de RH. Adicionalmente,
o SMOTE aplicado na secção 9.1 gera pontos sintéticos por interpolação no
espaço original das features; transformar esse espaço com PCA tornaria os
pontos sintéticos menos interpretáveis e mais difíceis de validar.

---
## 7. Definir as Features para o Modelo

In [11]:
"""As features numéricas estão prontas a usar directamente pelo modelo.
Já foram imputadas com a mediana e têm escala numérica contínua,
pelo que não precisam de qualquer transformação adicional nesta fase."""

features_num = ['indice_desenvolvimento_cidade', 'horas_formacao', 'anos_exp_num']
# lista das 3 colunas numéricas que entram directamente no modelo sem transformação

""" As features categóricas precisam de ser convertidas para números antes
de entrarem no modelo. A estratégia de encoding difere consoante o
algoritmo: Label Encoding para árvores de decisão, que não interpretam
a ordem dos inteiros, e One-Hot Encoding para a regressão logística,
que é sensível à ordem implícita dos valores inteiros atribuídos."""

features_cat = ['genero', 'experiencia_relevante', 'inscricao_universidade',
                'nivel_educacao', 'area_formacao', 'tipo_empresa',
                'dimensao_empresa', 'anos_desde_ultimo_emprego']
# lista das 8 colunas categóricas que precisam de conversão para número

print('Features numéricas:', features_num)                           # confirma as colunas numéricas
print('Features categóricas:', features_cat)                         # confirma as colunas categóricas

Features numéricas: ['indice_desenvolvimento_cidade', 'horas_formacao', 'anos_exp_num']
Features categóricas: ['genero', 'experiencia_relevante', 'inscricao_universidade', 'nivel_educacao', 'area_formacao', 'tipo_empresa', 'dimensao_empresa', 'anos_desde_ultimo_emprego']


### Análise

As features foram divididas em dois grupos: 3 numéricas prontas a usar
directamente e 8 categóricas que precisam de conversão para número antes
de entrarem no modelo. O encoding é feito na célula seguinte.

---
## 8. Encoding das Variáveis Categóricas

Os modelos de machine learning só trabalham com números. Por isso temos de converter as variáveis categóricas.

- **Label Encoding**: atribui um número a cada categoria (ex: "Masculino"=1, "Feminino"=0). Adequado para Árvores de Decisão.
- **One-Hot Encoding**: cria uma coluna binária por cada categoria. Adequado para Regressão Logística.

In [12]:
""" Construímos o dataset de trabalho com apenas as features seleccionadas
e a variável alvo. Este passo evita que colunas intermédias criadas
durante a preparação, como a 'anos_experiencia' original em texto,
contaminem os dados que entram nos modelos."""

# Preparar dataset base (sem as colunas originais que vamos substituir)

df_model = df[features_num + features_cat + ['target']].copy()
# features_num + features_cat = todas as features seleccionadas
# + ['target'] = acrescenta a variável alvo
# .copy() garante que alterações posteriores não afectam o df original

"""O Label Encoding atribui um inteiro a cada categoria de forma arbitrária.
É adequado para modelos baseados em árvore porque estes não interpretam
a ordem dos inteiros como distância matemática; dividem os dados por
limiares sem assumir relações lineares entre os valores codificados."""

# --- Label Encoding (para Árvore de Decisão) ---
le = LabelEncoder()                                                  # inicializa o encoder do scikit-learn
df_le = df_model.copy()                                              # cópia para não alterar df_model
for col in features_cat:                                             # percorre cada coluna categórica
    df_le[col + '_enc'] = le.fit_transform(df_le[col].astype(str))   # cria coluna nova com sufixo _enc
    # .astype(str) garante que 'Desconhecido' e outros valores são tratados como texto
    # fit_transform aprende as categorias e converte para inteiro em simultâneo

features_le = features_num + [c + '_enc' for c in features_cat]      # lista final de features para Label Encoding
print(f'Label Encoding: {len(features_le)} features')                # confirma o número de features

"""O One-Hot Encoding cria uma coluna binária por cada categoria, eliminando
qualquer ordem numérica implícita entre elas. É obrigatório para regressão
logística porque este algoritmo assume relações lineares; usar Label Encoding
criaria distâncias ordinais falsas entre categorias sem ordem natural.
O drop_first=True remove uma coluna de cada variável para evitar
multicolinearidade perfeita entre as dummies criadas."""

# --- One-Hot Encoding (para Regressão Logística) ---
df_ohe = pd.get_dummies(df_model, columns=features_cat, drop_first=True)
# pd.get_dummies cria automaticamente colunas binárias para cada categoria
# drop_first=True remove a primeira categoria de cada variável (referência)

features_ohe = [c for c in df_ohe.columns if c != 'target']         # lista de features sem a variável alvo

print(f'One-Hot Encoding: {len(features_ohe)} features')

print(f'\nPrimeiras colunas OHE: {features_ohe[:8]}')               # mostra as primeiras 8 colunas criadas

Label Encoding: 11 features
One-Hot Encoding: 41 features

Primeiras colunas OHE: ['indice_desenvolvimento_cidade', 'horas_formacao', 'anos_exp_num', 'genero_Feminino', 'genero_Masculino', 'genero_Outro', 'experiencia_relevante_Sem experiência relevante', 'inscricao_universidade_Curso a tempo parcial']


### Análise

O encoding foi aplicado com duas estratégias distintas.

O **Label Encoding** gerou 11 features, uma por cada variável (3 numéricas
já existentes e 8 categóricas convertidas para inteiro). Será usado com
a Árvore de Decisão.

O **One-Hot Encoding** gerou 41 features, porque cada categoria de cada
variável originou uma coluna binária própria (com excepção da primeira
categoria de cada variável, removida pelo **drop_first=True** para evitar
multicolinearidade). Será usado com a Regressão Logística.

As primeiras colunas do dataset OHE confirmam o padrão esperado: as 3
colunas numéricas mantêm-se inalteradas e as categóricas são expandidas
em colunas binárias como **genero_Feminino**, **genero_Masculino** e
**experiencia_relevante_Sem experiência relevante**.

---
## 9. Dividir em Treino e Teste

In [13]:
"""Extraímos a variável alvo que é partilhada pelas duas versões do dataset.
Tanto o conjunto com Label Encoding como o com One-Hot Encoding usam
exactamente o mesmo y, garantindo que os modelos são comparados sobre
as mesmas observações de treino e de teste."""

# Variável alvo
y = df_model['target']                                               # variável alvo partilhada pelos dois datasets

""" Dividimos o dataset com Label Encoding em 80% treino e 20% teste.
O parâmetro stratify=y garante que a proporção entre as classes é
preservada em ambas as partições; com um target desbalanceado isto é
essencial para que o teste tenha a mesma distribuição que o treino.
O random_state=SEED assegura que a divisão é idêntica entre execuções."""

# --- Para Label Encoding (Árvore de Decisão) ---
X_le = df_le[features_le]                                           # selecciona apenas as features do Label Encoding
X_tr_le, X_te_le, y_train, y_test = train_test_split(
    X_le, y, test_size=0.2, random_state=SEED, stratify=y
)
# test_size=0.2 = 20% dos dados vão para teste, 80% para treino
# random_state=SEED = garante que a divisão é sempre igual
# stratify=y = preserva a proporção entre classes em treino e teste

""" Aplicamos exactamente os mesmos parâmetros de divisão ao dataset com
One-Hot Encoding. Usar a mesma semente e o mesmo stratify garante que
os índices de treino e teste são idênticos aos do Label Encoding,
tornando a comparação entre os dois modelos directa e sem viés de amostragem."""

# --- Para One-Hot Encoding (Regressão Logística) ---
X_ohe = df_ohe[features_ohe]                                        # selecciona apenas as features do One-Hot Encoding
X_tr_ohe, X_te_ohe, _, _ = train_test_split(
    X_ohe, y, test_size=0.2, random_state=SEED, stratify=y
)
# _ , _ = descarta y_train e y_test porque já foram guardados na divisão anterior
print(f'✅ Split concluído:')
print('Divisão treino/teste (80% / 20% — estratificada):')
print(f'  Treino: {len(y_train):,} amostras | positivos: {y_train.sum():,} ({y_train.mean()*100:.1f}%)')
print(f'  Teste:  {len(y_test):,} amostras  | positivos: {y_test.sum():,} ({y_test.mean()*100:.1f}%)')
# positivos = candidatos que procuram mudar de emprego (target = 1)
# a percentagem deve ser semelhante nos dois conjuntos graças ao stratify

✅ Split concluído:
Divisão treino/teste (80% / 20% — estratificada):
  Treino: 15,326 amostras | positivos: 3,822 (24.9%)
  Teste:  3,832 amostras  | positivos: 955 (24.9%)


### Análise

O dataset foi dividido em 80% para treino (15 326 amostras) e 20% para
teste (3 832 amostras). A proporção de candidatos que procuram mudar de
emprego é exactamente igual nos dois conjuntos (24.9%), o que confirma
que o **stratify=y** funcionou correctamente. Esta consistência é essencial
para que a avaliação do modelo no teste seja representativa da realidade.

---
## 9.1 Balanceamento de Classes com SMOTE

O dataset tem uma distribuição desequilibrada: **75.1% não procura mudar** de emprego e **24.9% procura**. Com esta proporção, os modelos tendem a favorecer a classe maioritária, resultando num recall baixo para a classe que nos interessa prever.

O **SMOTE** (Synthetic Minority Oversampling Technique) resolve este problema criando observações sintéticas da classe minoritária por interpolação entre exemplos reais existentes, em vez de simplesmente duplicar registos. O resultado é um conjunto de treino equilibrado sem perda de informação.

**Regra fundamental:** o SMOTE é aplicado **apenas ao conjunto de treino**. O conjunto de teste mantém a distribuição original porque representa a realidade que o modelo vai encontrar em produção.

In [14]:
"""Aplicamos o SMOTE ao conjunto de treino para equilibrar as classes.
O parâmetro k_neighbors=5 define quantos vizinhos próximos são usados
para gerar cada ponto sintético; o valor padrão é 5 e é adequado para
este dataset. O random_state=SEED garante reprodutibilidade."""

smote = SMOTE(random_state=SEED,                                     # reprodutibilidade: a mesma semente gera sempre os mesmos sintéticos
              k_neighbors=5)                                         # usa os 5 vizinhos mais próximos para gerar cada ponto sintético

# ── Label Encoding (Árvore de Decisão) ──────────────────────────────────────
X_tr_le_smote, y_train_smote = smote.fit_resample(X_tr_le, y_train)
# fit_resample aprende a distribuição do treino e cria exemplos sintéticos da classe minoritária
# devolve X e y já equilibrados — nunca aplicar ao conjunto de teste

# ── One-Hot Encoding (Regressão Logística) ───────────────────────────────────
X_tr_ohe_smote, _ = smote.fit_resample(X_tr_ohe, y_train)
# mesmo processo para o conjunto OHE; y_train_smote é o mesmo, só X muda

# ── Verificar resultados ─────────────────────────────────────────────────────
print('=== ANTES DO SMOTE (treino) ===')
print(pd.Series(y_train).value_counts().rename({1: 'Procura (1)', 0: 'Não procura (0)'}))
print(f'Proporção classe positiva: {y_train.mean():.1%}')

print('\n=== DEPOIS DO SMOTE (treino) ===')
print(pd.Series(y_train_smote).value_counts().rename({1: 'Procura (1)', 0: 'Não procura (0)'}))
print(f'Proporção classe positiva: {y_train_smote.mean():.1%}')      # deve ser agora 50%

print('\n=== TESTE (inalterado) ===')
print(pd.Series(y_test).value_counts().rename({1: 'Procura (1)', 0: 'Não procura (0)'}))
print(f'Proporção classe positiva: {y_test.mean():.1%}')             # mantém a distribuição real

print(f'\nShape treino LE  antes: {X_tr_le.shape}  →  depois: {X_tr_le_smote.shape}')
print(f'Shape treino OHE antes: {X_tr_ohe.shape}  →  depois: {X_tr_ohe_smote.shape}')
print(f'Shape teste LE  (inalterado): {X_te_le.shape}')

=== ANTES DO SMOTE (treino) ===
target
Não procura (0)    11504
Procura (1)         3822
Name: count, dtype: int64
Proporção classe positiva: 24.9%

=== DEPOIS DO SMOTE (treino) ===
target
Não procura (0)    11504
Procura (1)        11504
Name: count, dtype: int64
Proporção classe positiva: 50.0%

=== TESTE (inalterado) ===
target
Não procura (0)    2877
Procura (1)         955
Name: count, dtype: int64
Proporção classe positiva: 24.9%

Shape treino LE  antes: (15326, 11)  →  depois: (23008, 11)
Shape treino OHE antes: (15326, 41)  →  depois: (23008, 41)
Shape teste LE  (inalterado): (3832, 11)


### Análise

O SMOTE equilibrou o conjunto de treino com sucesso.

**Antes do SMOTE:** 11 504 registos da classe 0 (Não procura) e 3 822 da classe 1 (Procura), numa proporção de 75.1% / 24.9%.

**Depois do SMOTE:** ambas as classes com 11 504 registos, numa proporção de 50% / 50%. Os novos registos da classe minoritária são sintéticos, gerados por interpolação entre exemplos reais existentes no conjunto de treino.

**O conjunto de teste permanece inalterado** com 2 877 registos da classe 0 e 955 da classe 1 (proporção 75.1% / 24.9%), representando a distribuição real. Avaliar com a distribuição original é essencial para que as métricas reflictam o desempenho efectivo do modelo em produção.

O efeito esperado nos modelos é um aumento do **Recall** da classe positiva, porque os algoritmos deixam de ser penalizados por verem proporcionalmente muito mais exemplos negativos durante o treino.

---
## 10. Guardar o Dataset Limpo

In [15]:
""" Guardamos o dataset limpo com as features seleccionadas e a variável alvo
em CSV para ser reutilizado nos notebooks seguintes sem repetir todo o
processo de preparação. Salvaguardar o estado intermédio é uma boa prática
em pipelines com múltiplas fases e facilita a depuração se algo correr mal."""

df_limpo = df_model.copy()                                           # cópia do dataset com as features finais
df_limpo.to_csv(os.path.join(OUTPUT_DIR, 'dataset_preparado.csv'), index=False)                     # guarda em CSV sem o índice do pandas

print(f'✅ Dataset limpo guardado: dataset_preparado.csv {df_limpo.shape}') # confirma o nome do ficheiro e as dimensões

✅ Dataset limpo guardado: dataset_preparado.csv (19158, 12)


### Análise

O dataset limpo foi guardado com sucesso em **dataset_preparado.csv** com
19 158 registos e 12 colunas (11 features e a variável alvo **target**).
Este ficheiro será carregado directamente nos notebooks seguintes,
evitando repetir todo o processo de preparação.

---
## 11. Conclusões desta Fase

Nesta fase preparámos os dados para o modelo:

- Removemos duplicados (0 encontrados) e as colunas **id_candidato** e **cidade**,
  sem valor preditivo para o modelo
- Confirmámos que todos os 19 158 registos têm variável alvo preenchida,
  pelo que nenhum foi removido
- Preenchemos os valores em falta: **mediana** para as variáveis numéricas
  (sem nulos neste caso) e **"Desconhecido"** para as 8 variáveis categóricas,
  totalizando 0 nulos após o tratamento
- Convertemos **anos_experiencia** de texto para número com a função **conv_exp()**,
  com "Mais de 20 anos" a ficar com o valor 21
- Criámos dois tipos de encoding: **Label Encoding** com 11 features
  (para a Árvore de Decisão) e **One-Hot Encoding** com 41 features
  (para a Regressão Logística)
- Dividimos os dados em **80% treino** (15 326 amostras) e **20% teste**
  (3 832 amostras), com estratificação que preservou a proporção de 24.9%
  de candidatos que procuram mudar de emprego em ambos os conjuntos
- Aplicámos **SMOTE** ao conjunto de treino para equilibrar as classes:
  de 11 504 vs 3 822 registos (75.1% / 24.9%) para 11 504 vs 11 504 (50% / 50%).
  O conjunto de teste manteve a distribuição original
- Guardámos o dataset limpo em **dataset_preparado.csv** com 19 158 registos
  e 12 colunas

**Próximo passo:** Fase 4: Modelling (`04_Modelling.ipynb`)